## Experiments with the Qwen-QwQ model series

With the aim of updating the repo to perform more general experiments on a wider range of open-source reasoning models

In [2]:
HF_MODEL_PATH = "Qwen/QwQ-32B"
DEVICE = "cuda:1"

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import os
import sys

/home/etheridge/adv-steer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(HF_MODEL_PATH, device_map=DEVICE, torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_PATH, use_fast=False)

# set model to eval mode
model.eval()

Loading checkpoint shards: 100%|██████████| 14/14 [00:11<00:00,  1.22it/s]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 5120)
    (layers): ModuleList(
      (0-63): 64 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=5120, out_features=5120, bias=True)
          (k_proj): Linear(in_features=5120, out_features=1024, bias=True)
          (v_proj): Linear(in_features=5120, out_features=1024, bias=True)
          (o_proj): Linear(in_features=5120, out_features=5120, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=5120, out_features=27648, bias=False)
          (up_proj): Linear(in_features=5120, out_features=27648, bias=False)
          (down_proj): Linear(in_features=27648, out_features=5120, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((5120,), eps=1e-05)
        (post_attention_layernorm): Qwen2RMSNorm((5120,), eps=1e-05)
      )
    )
    (norm): Qwen2RMSNorm((5120,), eps=1e-05)
    (rotary_emb

In [4]:

prompt = "A body is moving in a circular path at a constant speed. Will its average velocity for a specific time period equal its final instantaneous velocity at the end of this period? Explain your answer using concepts of physics and mathematics."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)



Okay, so the question is asking whether the average velocity of a body moving in a circular path at constant speed will equal its final instantaneous velocity at the end of a specific time period. Hmm, let me think about this step by step. 

First, I need to recall the definitions of average velocity and instantaneous velocity. Average velocity is the displacement divided by the total time taken, right? While instantaneous velocity is the rate of change of position at a particular instant, which includes both speed and direction. Since the body is moving in a circular path at constant speed, its speed isn't changing, but its direction is. That means the velocity is changing because velocity is a vector quantity dependent on direction.

Let me consider an example. Suppose the body is moving in a circle of radius r. Let's say the time period we're looking at is the time it takes to complete a full revolution. In that case, the displacement would be zero because it ends up where it starte

In [5]:
# print intermediate embedding dimensions
for name, param in model.named_parameters():
    if "weight" in name:
        print(f"{name}: {param.shape}")


model.embed_tokens.weight: torch.Size([152064, 5120])
model.layers.0.self_attn.q_proj.weight: torch.Size([5120, 5120])
model.layers.0.self_attn.k_proj.weight: torch.Size([1024, 5120])
model.layers.0.self_attn.v_proj.weight: torch.Size([1024, 5120])
model.layers.0.self_attn.o_proj.weight: torch.Size([5120, 5120])
model.layers.0.mlp.gate_proj.weight: torch.Size([27648, 5120])
model.layers.0.mlp.up_proj.weight: torch.Size([27648, 5120])
model.layers.0.mlp.down_proj.weight: torch.Size([5120, 27648])
model.layers.0.input_layernorm.weight: torch.Size([5120])
model.layers.0.post_attention_layernorm.weight: torch.Size([5120])
model.layers.1.self_attn.q_proj.weight: torch.Size([5120, 5120])
model.layers.1.self_attn.k_proj.weight: torch.Size([1024, 5120])
model.layers.1.self_attn.v_proj.weight: torch.Size([1024, 5120])
model.layers.1.self_attn.o_proj.weight: torch.Size([5120, 5120])
model.layers.1.mlp.gate_proj.weight: torch.Size([27648, 5120])
model.layers.1.mlp.up_proj.weight: torch.Size([2764

In [14]:
HF_MODEL_PATH = "Qwen/QwQ-16B"

subdirs = ["activations", "attack_results", "cautious_dir", "dataset"]
hf_path = os.path.join(HF_MODEL_PATH.split("/")[0], HF_MODEL_PATH.split("/")[1])

for subdir in subdirs:
    dir_path = os.path.join(hf_path, subdir)
    # create directory if it does not exist
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)
        print(f"Created directory: {dir_path}")

# tree 
os.system(f"tree . -L 4")

.
├── Qwen
│   ├── QwQ-16B
│   │   ├── activations
│   │   ├── attack_results
│   │   ├── cautious_dir
│   │   └── dataset
│   └── QwQ-32B
│       ├── activations
│       ├── attack_results
│       ├── cautious_dir
│       └── dataset
└── qwen.ipynb

12 directories, 1 file


0

In [31]:
sys.path.append(os.path.dirname(os.getcwd()))
import utils.paths

utils.paths.validate_hf_id("sdf//fdsdfdsf")

AssertionError: Hugging Face ID should be in the format 'org_name/model_name'.